# EchoFactory - PUMP: Preprocessing & Feature Extraction
Extract 2 fitur dari raw .wav untuk STgram-MFN pada mesin `pump`:
- **Branch 0**: Log-Mel Spectrogram (n_fft=1024)
- **Branch 1**: Tgram - Tangential gram (n_fft=512)


In [ ]:
import os
import gc
import glob
import json
import numpy as np
import librosa
import torch
import torch.nn.functional as F
from tqdm import tqdm

print(f'torch {torch.__version__} | librosa {librosa.__version__}')


In [ ]:
DATASET_ROOT = '/kaggle/input/datasets/bisheshgiri/mimii-dataset'
OUT_DIR = '/kaggle/working'
TARGET_SNR = '0_dB'
MACHINE_TYPE = 'pump'
SR = 16000
IMG_SIZE = (128, 128)

MEL_N_MELS, MEL_N_FFT, MEL_HOP = 128, 1024, 512
TG_N_MELS, TG_N_FFT, TG_HOP = 128, 512, 512

os.makedirs(f'{OUT_DIR}/features', exist_ok=True)
print(f'Config OK | Machine: {MACHINE_TYPE} | Dataset: {DATASET_ROOT}')


In [ ]:
def compute_mel(wav):
    mel = librosa.feature.melspectrogram(y=wav, sr=SR, n_mels=MEL_N_MELS, n_fft=MEL_N_FFT, hop_length=MEL_HOP)
    return librosa.power_to_db(mel, ref=np.max)

def compute_tgram(wav):
    mel = librosa.feature.melspectrogram(y=wav, sr=SR, n_mels=TG_N_MELS, n_fft=TG_N_FFT, hop_length=TG_HOP)
    return librosa.power_to_db(mel, ref=np.max)

def normalize(x):
    return (x - x.mean()) / (x.std() + 1e-8)

def to_tensor(x_np):
    t = torch.FloatTensor(x_np).unsqueeze(0).unsqueeze(0)
    return F.interpolate(t, size=IMG_SIZE, mode='bilinear', align_corners=False).squeeze(0)

def extract_features(wav_path):
    wav = librosa.load(wav_path, sr=SR, mono=True)[0]
    mel_feat = to_tensor(normalize(compute_mel(wav)))
    tg_feat = to_tensor(normalize(compute_tgram(wav)))
    return torch.cat([mel_feat, tg_feat], dim=0)


In [ ]:
mpath = os.path.join(DATASET_ROOT, f'0_dB_pump', 'pump')
if os.path.exists(mpath):
    ids = sorted(os.listdir(mpath))
    label_map = {mid: i for i, mid in enumerate(ids)}
    print(f'Label map {MACHINE_TYPE}: {label_map}')
    with open(f'/kaggle/working/label_map_pump.json', 'w') as f:
        json.dump(label_map, f, indent=2)


In [ ]:
if os.path.exists(mpath):
    print(f'\nProcessing {MACHINE_TYPE.upper()}...')
    for condition in ['normal', 'abnormal']:
        all_feats, all_labels, all_paths = [], [], []
        for mid, label_id in label_map.items():
            cp = os.path.join(mpath, mid, condition)
            if not os.path.exists(cp): continue
            files = sorted(glob.glob(os.path.join(cp, '*.wav')))
            print(f'  {mid}/{condition}: {len(files)} files')
            for fp in tqdm(files, desc=mid, leave=False):
                try:
                    all_feats.append(extract_features(fp))
                    all_labels.append(label_id)
                    all_paths.append(fp)
                except Exception as e:
                    print(f'  Skip {os.path.basename(fp)}: {e}')

        if all_feats:
            save_path = f'/kaggle/working/features/{MACHINE_TYPE}_{condition}.pt'
            torch.save({
                'features': torch.stack(all_feats),
                'labels': torch.LongTensor(all_labels),
                'paths': all_paths
            }, save_path)
            mb = os.path.getsize(save_path) / (1024 ** 2)
            print(f'  SAVED: {save_path} | Shape: {torch.stack(all_feats).shape} | {mb:.1f} MB')
            del all_feats, all_labels, all_paths
            gc.collect()
